# British Airways Reviews — Data Collection & Cleaning

**Goal:** Scrape British Airways customer reviews from [Skytrax](https://www.airlinequality.com), clean the raw text, and save a processed dataset ready for downstream analysis (e.g., sentiment analysis, topic modelling).

## Notebook Structure
1. [Setup & Imports](#1-setup--imports)
2. [Web Scraping](#2-web-scraping)
3. [Save Raw Data](#3-save-raw-data)
4. [Load & Inspect Data](#4-load--inspect-data)
5. [Text Preprocessing](#5-text-preprocessing)
6. [Apply Pipeline & Save Clean Data](#6-apply-pipeline--save-clean-data)
7. [Quick Visual Check — Word Cloud](#7-quick-visual-check--word-cloud)

---
## 1. Setup & Imports

In [ ]:
# Install NLTK if not already available (only needs to run once)
!pip install nltk

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import re

# ── HTTP & HTML parsing ───────────────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import wordcloud as wc

# ── Profiling (optional) ──────────────────────────────────────────────────────
# import ydata_profiling as yp  # Uncomment for automated EDA report

# Download required NLTK data (run once, then comment out)
# nltk.download('punkt')
# nltk.download('stopwords')

---
## 2. Web Scraping

Scrape British Airways reviews from Skytrax (`airlinequality.com`).
Each page returns up to 100 reviews; we iterate over all available pages.

> **Note:** The scraper requires an active internet connection.  
> If the data is already downloaded, skip to **Section 4**.

In [ ]:
# ── Scraping configuration ────────────────────────────────────────────────────
BASE_URL  = "https://www.airlinequality.com/airline-reviews/british-airways"
PAGES     = 36     # Total pages to scrape
PAGE_SIZE = 100    # Reviews per page

reviews = []   # Will hold the raw review text for every review
ratings  = []  # Will hold the overall star rating for every review

for page_num in range(1, PAGES + 1):
    print(f"Scraping page {page_num} …")

    # Build the paginated URL (sorted by newest first)
    url = f"{BASE_URL}/page/{page_num}/?sortby=post_date%3ADesc&pagesize={PAGE_SIZE}"

    # Fetch the HTML content
    response = requests.get(url)
    soup     = BeautifulSoup(response.content, "html.parser")

    # ── Extract review text ───────────────────────────────────────────────────
    # Each review body lives inside a <div class="text_content"> element
    for paragraph in soup.find_all("div", {"class": "text_content"}):
        reviews.append(paragraph.get_text())

    # ── Extract star ratings ──────────────────────────────────────────────────
    # All rating elements (overall + sub-scores) share the "rating-10" class.
    # Index 0 = overall rating; indices 1…PAGE_SIZE = per-review overall ratings.
    rating_elements = soup.find_all("div", {"class": "rating-10"})
    print(f"  Found {len(rating_elements)} rating elements on this page")

    for idx in range(1, PAGE_SIZE + 1):
        # The <span itemprop="ratingValue"> contains the numeric score
        rating_span = rating_elements[idx].find("span", {"itemprop": "ratingValue"})
        if rating_span is not None:
            ratings.append(int(rating_span.get_text().strip()))
        else:
            # Missing rating — store NaN so we can handle it later
            ratings.append(np.nan)

    print(f"  → {len(reviews)} reviews | {len(ratings)} ratings collected so far")

print("\n✅ Scraping complete!")

In [ ]:
# Sanity check — both lists must have the same length
print(f"Total reviews : {len(reviews)}")
print(f"Total ratings : {len(ratings)}")
assert len(reviews) == len(ratings), "Mismatch between reviews and ratings!"

---
## 3. Save Raw Data

In [ ]:
# Combine into a DataFrame and persist to disk
df_raw = pd.DataFrame({"reviews": reviews, "ratings": ratings})

# Quick preview
print(df_raw.shape)
df_raw.head()

In [ ]:
# Save raw dataset (relative path from the notebooks/ folder)
df_raw.to_csv("../data/BA_reviews.csv", index=False)
print("Raw data saved to data/BA_reviews.csv")

---
## 4. Load & Inspect Data

Load the raw CSV (useful when re-running the notebook without re-scraping),
then perform basic quality checks.

In [2]:
# Load the raw CSV
df = pd.read_csv("../data/BA_reviews.csv")

# Reset index to ensure it's a clean RangeIndex
df.index = range(df.shape[0])

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (3600, 3)


,Unnamed: 0,reviews,ratings
0,0,Not Verified | British Airways has confirmed ...,1.0
1,1,✅ Trip Verified | Worst BA experience. I was s...,2.0
2,2,✅ Trip Verified | My daughter and I were deni...,1.0
3,3,✅ Trip Verified | Despite boarding being the u...,8.0
4,4,"Not Verified | Flight cancelled, no crew! 9th...",1.0


In [3]:
# ── Remove duplicate rows ─────────────────────────────────────────────────────
before = df.shape[0]
df.drop_duplicates(inplace=True)
df.index = range(df.shape[0])   # Re-index after dropping
print(f"Removed {before - df.shape[0]} duplicate row(s).  Remaining: {df.shape[0]}")

Removed 0 duplicate row(s).  Remaining: 3600


In [4]:
# ── Inspect missing values ────────────────────────────────────────────────────
print("Missing values per column:")
print(df.isna().sum())

Missing values per column:
Unnamed: 0    0
reviews       0
ratings       5
dtype: int64


In [5]:
# Drop rows that have no rating (cannot be used for supervised learning)
df.dropna(inplace=True)
df.index = range(df.shape[0])   # Re-index after dropping

print("Missing values after dropping NaNs:")
print(df.isna().sum())

Missing values after dropping NaNs:
Unnamed: 0    0
reviews       0
ratings       0
dtype: int64


In [6]:
# Structural overview of the cleaned DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  3595 non-null   int64  
 1   reviews     3595 non-null   object 
 2   ratings     3595 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 84.4+ KB


In [7]:
# ── Check verified vs. unverified trips ──────────────────────────────────────
# Skytrax marks trip-verified reviews with "Trip Verified" in the text.
verified_count = np.array(
    ["Trip Verified" in df["reviews"][i] for i in range(df.shape[0])]
).sum()

print(f"Verified trips   : {verified_count}")
print(f"Unverified trips : {df.shape[0] - verified_count}")

Verified trips   : 1118
Unverified trips : 2477


---
## 5. Text Preprocessing

We build a modular cleaning pipeline with four steps:

| Step | Function | Description |
|------|----------|-------------|
| 1 | `links_remover` | Strip `http://` and `https://` URLs |
| 2 | `special_char_remover` | Remove non-alphabetic special characters |
| 3 | `contraction_expansion` | Expand English contractions (e.g., *won't → would not*) |
| 4 | `stopwords_remover` | Drop stop words and keep only alphabetic tokens |

In [8]:
# ── Stopword list ─────────────────────────────────────────────────────────────
# Start with NLTK's English stopword list, then customise it.
nltk_stop_words = stopwords.words("english")

# Add domain-specific words that add noise for BA review analysis
domain_words = [
    "flight", "british", "airway", "airways",  # airline-related words
    "ba", "airline",                           # brand abbreviations
    "seat", "seats",                            # very common & uninformative alone
    "one", "trip", "verified",                  # Skytrax artefacts
]
nltk_stop_words.extend(domain_words)

# Keep negation words — they are critical for sentiment analysis
for negation in ["not", "no"]:
    if negation in nltk_stop_words:
        nltk_stop_words.remove(negation)

print(f"Total stopwords: {len(nltk_stop_words)}")
print("Sample:", nltk_stop_words[:10])

Total stopwords: 188
Sample: ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're"]


In [9]:
# ── Cleaning helper functions ─────────────────────────────────────────────────

def links_remover(text: str) -> str:
    """Remove HTTP and HTTPS URLs from text."""
    text = re.sub(r"http\S+", "", text)   # Remove http://... links
    text = re.sub(r"https\S+", "", text)  # Remove https://... links
    return text


def special_char_remover(text: str) -> str:
    """Strip special characters, leaving only letters, digits and whitespace."""
    return re.sub(r"[^\w\s]", "", text)   # \w = [a-zA-Z0-9_], \s = whitespace


def contraction_expansion(text: str) -> str:
    """Expand common English contractions to their full forms.

    Expanding contractions before stopword removal ensures that
    'won't' → 'would not' keeps the negation 'not' in the text.
    """
    # Order matters: longer/more-specific patterns must come first
    contractions = [
        (r"won\'t",    "would not"),
        (r"can\'t",    "can not"),
        (r"don\'t",    "do not"),
        (r"shouldn\'t", "should not"),
        (r"needn\'t",   "need not"),
        (r"hasn\'t",    "has not"),
        (r"weren\'t",   "were not"),
        (r"didn\'t",    "did not"),
        (r"n\'t",       "not"),    # Generic negative (e.g., isn't, wasn't)
        (r"\'re",       "are"),
        (r"\'s",        "is"),
        (r"\'ll",       "will"),
        (r"\'t",        "not"),
        (r"\'ve",       "have"),
        (r"\'m",        "am"),
    ]
    for pattern, replacement in contractions:
        text = re.sub(pattern, replacement, text)
    return text


def stopwords_remover(text: str) -> str:
    """Tokenise by whitespace, remove stopwords and non-alpha tokens.

    Tokens are lowercased and only kept if they:
      - are purely alphabetic (removes numbers and punctuation remnants)
      - are not in the stopword list
    """
    tokens = [
        token.strip().lower()
        for token in text.split()
        if token.strip().lower().isalpha()             # keep alphabetic only
        and token.strip().lower() not in nltk_stop_words  # drop stopwords
    ]
    return " ".join(tokens)


def data_cleaner(text: str) -> str:
    """Full cleaning pipeline applied to a single review string.

    Pipeline order:
        1. Remove URLs
        2. Remove special characters
        3. Expand contractions  ← must happen before stopword removal
           so negations (not, no) survive
        4. Remove stopwords & non-alpha tokens
    """
    text = links_remover(text)
    text = special_char_remover(text)
    text = contraction_expansion(text)
    text = stopwords_remover(text)
    return text

---
## 6. Apply Pipeline & Save Clean Data

In [10]:
# Keep the original text for reference / auditing
df.rename(columns={"reviews": "original_review"}, inplace=True)

In [11]:
%%time
# Apply the full cleaning pipeline to every review
# %%time reports wall-clock time (typically ~1-2 s for ~3 600 rows)
df["clean_review"] = df["original_review"].apply(data_cleaner)

CPU times: total: 2.22 s
Wall time: 3.01 s


In [14]:
# Reorder columns for readability
df = df[["original_review", "clean_review", "ratings"]]

print(f"Final dataset shape: {df.shape}")
df.head(10)

Final dataset shape: (3595, 3)


,original_review,clean_review,ratings
0,Not Verified | British Airways has confirmed ...,not confirmed worst uk last minute cancellatio...,1.0
1,✅ Trip Verified | Worst BA experience. I was s...,worst experience supposed fly italy september ...,2.0
2,✅ Trip Verified | My daughter and I were deni...,daughter denied boarding business class madrid...,1.0
3,✅ Trip Verified | Despite boarding being the u...,despite boarding usual free lhr groups called ...,8.0
4,"Not Verified | Flight cancelled, no crew! 9th...",not cancelled no crew september not peak holid...,1.0
5,"Not Verified | The worst service ever, my bag...",not worst service ever baggage not arrive time...,1.0
6,✅ Trip Verified | 4/4 flights we booked this ...,flights booked holiday delayed hours no wifi a...,1.0
7,✅ Trip Verified | British Airways has a total...,total lack respect customers booked fly frankf...,1.0
8,"✅ Trip Verified | London Heathrow to Keflavik,...",london heathrow keflavik iceland business clas...,8.0
9,✅ Trip Verified | Mumbai to London Heathrow in...,mumbai london heathrow business class ageing a...,8.0


In [12]:
# Save the clean dataset to the outputs folder
df.to_csv("../data/clean_BA_reviews.csv", index=False)
print("Clean data saved to data/clean_BA_reviews.csv")

Clean data saved to data/clean_BA_reviews.csv


---
## 7. Quick Visual Check — Word Cloud

A word cloud on the cleaned reviews gives a fast intuitive summary of the most
frequent terms. Words already removed by the pipeline should not appear.

In [13]:
# ── Word cloud configuration ──────────────────────────────────────────────────
# Combine all clean reviews into a single string for the generator
all_clean_text = " ".join(df["clean_review"].tolist())

# Extra words to suppress in the cloud (Skytrax artefacts still present after cleaning)
cloud_extra_stopwords = nltk_stop_words + ["no", "not", "trip", "ba", "verified"]

cloud = wc.WordCloud(
    width=1200,
    height=800,
    stopwords=cloud_extra_stopwords,
    background_color="white",
    colormap="viridis",
    max_words=200,
)

cloud.generate(all_clean_text)

# Display using Plotly for an interactive widget
reviews_cloud = px.imshow(cloud, width=900, height=450, title="Most Frequent Terms in BA Reviews")
reviews_cloud.update_layout(coloraxis_showscale=False)
reviews_cloud.show()

---
## 8. Summary

This notebook collects British Airways customer reviews from Skytrax, checks and cleans the review data, and saves a processed dataset for further analysis.

### What was done
- Scraped review text and associated ratings, with a check that the two collections have matching lengths.
- Inspected the data for duplicates and missing values. In the current run, 3,600 rows were loaded; five rows with missing ratings were removed, leaving 3,595 reviews.
- Preserved the original review text and created a cleaned version by removing URLs and special characters, expanding contractions, lowercasing tokens, and removing stopwords while retaining negations such as “not” and “no”.
- Saved the cleaned data with `original_review`, `clean_review`, and `ratings` columns to `data/clean_BA_reviews.csv`.
- Used a word cloud as a quick visual check of the cleaned review vocabulary.

### Output and handoff
The cleaned CSV is the input for the EDA and sentiment-analysis notebook. Scraping requires an internet connection; when the raw CSV is already available, start at the data-loading section rather than re-scraping.

### Note on the results
The row counts above reflect the currently saved notebook outputs. They may differ if the source website changes or the scraping configuration is changed.